In [1]:
import os
import rasterio
import numpy as np
import geopandas as gpd
import imageio

from PIL import Image
from rasterio.features import shapes
from shapely.geometry import shape, LineString
from skimage.morphology import medial_axis
from tqdm import tqdm

In [2]:
# -------------------------
# 📂 INPUTS
# -------------------------
raster_path = "data/el_harrach_georef.tif"
legend_dir = "data/legend_line_clean"

output_dir = "output/vect/line"
os.makedirs(output_dir, exist_ok=True)

BG_COLOR = np.array([241, 238, 232])
BG_TOLERANCE = 10
COLOR_TOLERANCE = 1

In [3]:
# -------------------------
# 🎨 LEGEND COLOR EXTRACTION
# -------------------------
def get_dominant_rgb(path):
    img = Image.open(path).convert("RGB")
    arr = np.array(img).reshape(-1, 3)

    mask = np.linalg.norm(arr - BG_COLOR, axis=1) > BG_TOLERANCE
    filtered = arr[mask]

    if filtered.size == 0:
        raise ValueError(f"No valid pixels in {path}")

    return np.median(filtered, axis=0).astype(np.uint8)


In [4]:
def build_color_class_map(legend_dir):
    color_map = {}

    files = [
        f for f in os.listdir(legend_dir)
        if f.lower().endswith(".png")
    ]

    for file in tqdm(files, desc="🎨 Extracting legend colors"):
        path = os.path.join(legend_dir, file)
        class_name = os.path.splitext(file)[0]

        rgb = get_dominant_rgb(path)
        color_map[class_name] = tuple(rgb)

        print(f"🎨 {class_name} → {rgb}")

    return color_map

In [5]:
color_map = build_color_class_map(legend_dir)

🎨 Extracting legend colors: 100%|██████████| 95/95 [00:00<00:00, 924.74it/s]

🎨 Minor_power_line___Path_from_tee_area_to_the_green_of_a_golf_course → [232 229 223]
🎨 Living_street → [232 233 232]
🎨 Access_road__may_be_also_outside_of_a_city → [254 254 254]
🎨 Living_street_under_construction → [199 197 195]
🎨 The_link_roads__sliproads___ramps__leading_to_and_from_a_trunk_highway → [249 178 156]
🎨 Sub-national_boundary__fourth-highest_level → [228 205 221]
🎨 Miniature_railway → [197 195 192]
🎨 River___Canal → [170 211 223]
🎨 Residential_road_only_local_traffic → [252 252 252]
🎨 Cycleway → [215 213 242]
🎨 Taxiway → [188 188 204]
🎨 Subordinated_way_in_a_parking_lot___drive-through_highway___driveway___slipway → [240 238 235]
🎨 Track__Solid_surface → [205 169  97]
🎨 River_intermittent___Canal_intermittent___River_seasonal___Canal_seasonal → [201 222 227]
🎨 Embankment → [220 218 213]
🎨 Trunks__the_most_important_roads_in_a_road_network_that_aren_t_motorways → [249 178 156]
🎨 Stream_in_pipe_or_tunnel___Ditch_in_pipe_or_tunnel___drain_in_pipe_or_tunnel → [221 232 232]
🎨

In [6]:
# -------------------------
# 📥 LOAD RASTER
# -------------------------
def load_raster(path):
    with rasterio.open(path) as src:
        img = src.read()
        transform = src.transform
        crs = src.crs

    img = np.transpose(img, (1, 2, 0))[:, :, :3].astype(np.uint8)
    return img, transform, crs


In [7]:
img, transform, crs = load_raster(raster_path)

In [8]:
# -------------------------
# 🎯 MASK
# -------------------------
def build_mask(img, rgb, tol):
    target = np.array(rgb).astype(np.int16)
    img_i16 = img.astype(np.int16)

    return np.all(np.abs(img_i16 - target) <= tol, axis=2)

In [9]:
# -------------------------
# 🧵 SKELETON → LINES
# -------------------------
def skeleton_to_lines(skel):
    lines = []
    visited = skel.copy()

    h, w = skel.shape

    for y in range(h):
        for x in range(w):

            if not skel[y, x] or not visited[y, x]:
                continue

            coords = []
            cy, cx = y, x

            while True:
                if not (0 <= cy < h and 0 <= cx < w):
                    break
                if not skel[cy, cx]:
                    break

                coords.append((cx, cy))
                visited[cy, cx] = False

                found = False
                for dy in [-1, 0, 1]:
                    for dx in [-1, 0, 1]:
                        ny, nx = cy + dy, cx + dx

                        if (
                            0 <= ny < h and 0 <= nx < w
                            and skel[ny, nx]
                            and visited[ny, nx]
                        ):
                            cy, cx = ny, nx
                            found = True
                            break
                    if found:
                        break

                if not found:
                    break

            if len(coords) > 3:
                lines.append(LineString(coords))

    return lines

In [10]:
# -------------------------
# 🌍 PIXEL → GEO
# -------------------------
def pixel_to_geo(coords, transform):
    return [
        rasterio.transform.xy(transform, y, x)
        for x, y in coords
    ]

In [11]:
def process_class(class_name, rgb, img, transform, crs):
    mask = build_mask(img, rgb, COLOR_TOLERANCE)

    # -------------------------
    # ❌ SKIP: empty mask early
    # -------------------------
    if mask is None or not np.any(mask):
        print(f"⏭️ SKIPPED (empty mask): {class_name}")
        return

    mask = mask.astype(bool)

    # -------------------------
    # 🧠 Skeleton extraction
    # -------------------------
    try:
        skel, _ = medial_axis(mask, return_distance=True)
    except Exception as e:
        print(f"⚠️ SKIPPED (medial_axis failed): {class_name} -> {e}")
        return

    lines = skeleton_to_lines(skel)

    if not lines:
        print(f"⏭️ SKIPPED (no skeleton lines): {class_name}")
        return

    geoms = []

    # -------------------------
    # 🧵 Line processing
    # -------------------------
    for line in tqdm(lines, desc=f"🧵 {class_name}", leave=False):
        if line is None:
            continue

        # basic geometry sanity check
        if line.is_empty or line.length == 0:
            continue

        coords = list(line.coords)

        if len(coords) < 2:
            continue

        # pixel → geo transform
        try:
            geo = pixel_to_geo(coords, transform)
        except Exception:
            continue

        if not geo or len(geo) < 2:
            continue

        # final validation before geometry creation
        try:
            geom = LineString(geo)
            if geom.is_empty or geom.length == 0:
                continue
            geoms.append(geom)
        except Exception:
            continue

    # -------------------------
    # ❌ SKIP: no valid geometries
    # -------------------------
    if len(geoms) == 0:
        print(f"⏭️ SKIPPED (no valid geometries): {class_name}")
        return

    # -------------------------
    # 💾 SAVE GEOJSON
    # -------------------------
    try:
        gdf = gpd.GeoDataFrame(geometry=geoms, crs=crs)
        gdf["class"] = class_name

        geojson_path = os.path.join(output_dir, f"{class_name}.geojson")
        gdf.to_file(geojson_path, driver="GeoJSON")

    except Exception as e:
        print(f"❌ FAILED saving GeoJSON: {class_name} -> {e}")
        return

    # -------------------------
    # 🖼️ SAVE OVERLAY
    # -------------------------
    try:
        overlay = img.copy()

        color = np.array([255, 0, 0], dtype=np.uint8)

        if overlay.shape[2] == 3:
            overlay[mask] = color
        elif overlay.shape[2] == 4:
            overlay[mask] = np.array([255, 0, 0, 255], dtype=np.uint8)

        overlay_path = os.path.join(output_dir, f"{class_name}_overlay.png")
        imageio.imwrite(overlay_path, overlay)

    except Exception as e:
        print(f"⚠️ Overlay save failed: {class_name} -> {e}")

    # -------------------------
    # ✅ SUCCESS
    # -------------------------
    print(f"✅ EXPORTED: {class_name}")

In [12]:
items = list(color_map.items())

for class_name, rgb in tqdm(items, desc="🚀 Processing classes"):
    process_class(class_name, rgb, img, transform, crs)

🚀 Processing classes:   1%|          | 1/95 [00:28<45:06, 28.79s/it]

✅ EXPORTED: Minor_power_line___Path_from_tee_area_to_the_green_of_a_golf_course


🚀 Processing classes:   2%|▏         | 2/95 [00:55<42:21, 27.33s/it]

✅ EXPORTED: Living_street


🚀 Processing classes:   3%|▎         | 3/95 [01:40<54:25, 35.49s/it]

...

⏭️ SKIPPED (no skeleton lines): Sub-national_boundary__fifth-highest_or_sixth-highest_level


🚀 Processing classes:  99%|█████████▉| 94/95 [34:58<00:23, 23.39s/it]

✅ EXPORTED: Roller_coaster_track


🚀 Processing classes: 100%|██████████| 95/95 [35:23<00:00, 22.36s/it]

✅ EXPORTED: Conveyor_system_for_transporting_materials
